# Helathcare Risk Factors Dataset

**Dataset**

Age: Patient’s age (in years).

Gender: Male / Female.

Medical Condition: Reported health condition (e.g., Diabetes, Hypertension, Asthma, Obesity, Healthy).

Glucose: Blood glucose level.

Blood Pressure: Blood pressure measurement.

BMI: Body Mass Index.

Oxygen Saturation: Blood oxygen saturation level.

LengthOfStay: Hospital length of stay (days).

Cholesterol: Cholesterol level.

Triglycerides: Triglyceride level.

HbA1c: Hemoglobin A1c (glycated hemoglobin).

Smoking: Smoking status (0 = Non-smoker, 1 = Smoker).

Alcohol: Alcohol consumption (0 = No, 1 = Yes).

Physical Activity: Physical activity (approx. hours/week).

Diet Score: Diet quality score (numeric).

Family History: Family medical history (0 = No, 1 = Yes).

Stress Level: Stress level (numeric scale).

Sleep Hours: Average sleep hours per day.

random_notes: Random notes

noise_col: Noise column (unrelated random values).

In [ ]:
# Install libraries
!pip install xgboost lightgbm catboost -qqq

import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.linear_model import (LogisticRegression, RidgeClassifier, SGDClassifier, Perceptron)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier, RadiusNeighborsClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier,
                              HistGradientBoostingClassifier, BaggingClassifier, VotingClassifier, StackingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.8 MB/s eta 0:00:00


In [ ]:
# Load the dataset

from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv('dirty_v3_path.csv')

In [ ]:
# Preview Data
df.sample(5)

In [ ]:
def analyze_columns_for_encoding(df, threshold=50):
    """
    Analyzes the dataframe columns, checking the number of unique values
    and recommending which columns should be dropped, kept, or treated
    before using get_dummies for encoding to avoid dimensionality issues.

    Parameters:
    df : pandas.DataFrame
        The dataframe to analyze.
    threshold : int
        The maximum number of unique values in a column to recommend encoding.

    Returns:
    report : pandas.DataFrame
        A report with recommendations for each column.
    """
    # Create an empty DataFrame for the report
    report = pd.DataFrame(columns=['Column', 'Data Type', 'Unique Values', 'Recommendation'])

    for column in df.columns:
        unique_values = df[column].nunique()
        data_type = df[column].dtype

        # Determine the recommendation based on unique values and data type
        if data_type == 'object' or data_type == 'category':
            if unique_values > threshold:
                recommendation = "Consider Dropping or Encoding Differently (Too many categories)"
            elif unique_values == 2:
                recommendation = "Keep (Binary category)"
            else:
                recommendation = "Keep but beware of dimensionality (Low unique categories)"
        elif data_type in ['int64', 'float64']:
            if unique_values == 2:
                recommendation = "Keep (Binary numeric)"
            else:
                recommendation = "Keep (Continuous)"
        else:
            recommendation = "Consider Dropping (Unknown data type)"

        # Create a new row with the column analysis
        new_row = pd.DataFrame({
            'Column': [column],
            'Data Type': [data_type],
            'Unique Values': [unique_values],
            'Recommendation': [recommendation]
        })

        # Concatenate the new row to the report DataFrame
        report = pd.concat([report, new_row], ignore_index=True)

    return report

# Set your threshold for handling categorical variables
threshold_value = 50

# Analyze the dataframe columns
report = analyze_columns_for_encoding(df, threshold=threshold_value)

# Display the report
print(report)

In [ ]:
df['target'].unique()


In [ ]:
# Create a new binary target column
# Map 'Healthy' to 0 and all other medical conditions to 1
df['target'] = df['target'].apply(lambda x: 0 if x == 'Healthy' else 1)

# Display the value counts of the new target column
print(df['target'].value_counts())

# Identify the Target column

In [ ]:
df = df.rename(columns={'orig_name': 'target'})
df.head()
# Identify the target column and ensure it's binary (0/1)
# If the column already contains two unique values, we map them to 0 and 1 if necessary

# Get unique values of the target column
unique_values = df['target'].unique()

# Check if the column has exactly two unique values
if len(unique_values) != 2:
    raise ValueError("Target column does not have exactly two unique values for binary classification.")
else:
    # If values are not already 0 and 1, map them to 0 and 1
    if set(unique_values) != {0, 1}:
        value_map = {unique_values[0]: 0, unique_values[1]: 1}
        df['target'] = df['target'].map(value_map)
        print(f"Target values have been mapped to 0 and 1: {value_map}")
    else:
        print("Target values are already in binary (0 and 1).")

# Display the target column and verify the changes
print(f"Target column: {df['target'].name}")
print(df['target'].unique())

# Step 2: Validate Data types and inspect dataset structure

In [ ]:
df.info()

# Handle Unique Values and Duplicates

In [ ]:
# Check the number of unique values per column
print(df.nunique())

# Remove duplicate rows
df = df.drop_duplicates()   # this may need to be a bit more complex based on your use case / data

# Check for balanced target

In [ ]:
# Check if the target class is balanced
print(df['target'].value_counts())

# Drop Unnecessary Columns

In [ ]:
# Drop non-numeric columns that do not contribute to prediction
df = df.drop(columns=['random_notes', 'noise_col'])
df.head()
df.info()
df.describe()

# Handle missing data

In [ ]:
# Check for missing values
print("Rows/Cols before: " + str(df.shape))
print(df.isnull().sum())

# Drop rows with missing values
df = df.dropna()
print("Rows/Cols after: " + str(df.shape))

In [ ]:
from scipy import stats
print("Rows/Cols before: " + str(df.shape))
#Remove outliers using Z-score
z_scores = stats.zscore(df.select_dtypes(include=['float64', 'int64']))
abs_z_scores = abs(z_scores)
df = df[(abs_z_scores < 3).all(axis=1)]
print("Rows/Cols after: " + str(df.shape))

# Encode categorical Variables

In [ ]:
df.info()

# Separate Features and Target, Then Perform Train-Test Split

In [ ]:
# Separate the features (X) and the target (y)
X = df.drop(columns=['target'])
y = df['target']

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and test sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

 # Build an Algorithm Harness to Compare Multiple Binary Classifiers

In [ ]:
# Define base classifiers for Voting and Stacking
base_classifiers = [
    ('lr', LogisticRegression()),
    ('rf', RandomForestClassifier()),
    ('svc', SVC(probability=True))  # Probability required for AUC-ROC
]

In [ ]:
# Combine all classifiers, comment out those you don't want to use
classifiers = [
    LogisticRegression(), RidgeClassifier(), SGDClassifier(), Perceptron(),
    SVC(probability=True), KNeighborsClassifier(), RadiusNeighborsClassifier(),
    GaussianProcessClassifier(), DecisionTreeClassifier(), RandomForestClassifier(),
    GradientBoostingClassifier(), AdaBoostClassifier(), HistGradientBoostingClassifier(),
    BaggingClassifier(),
    VotingClassifier(estimators=base_classifiers, voting='soft'),  # Voting Classifier with base classifiers
    StackingClassifier(estimators=base_classifiers),  # Stacking Classifier with base classifiers
    MLPClassifier(),
    GaussianNB(),
    xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss'), lgb.LGBMClassifier(), CatBoostClassifier(verbose=0)]

# Combine ALL classifiers

In [ ]:
# DataFrame to store ROC-AUC scores
results_df = pd.DataFrame(columns=['Model', 'ROC-AUC Score'])

# Iterate over models and evaluate performance
for model in classifiers:
    try:
        start = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - start

        start = time.time()
        y_pred = model.predict(X_test)
        y_pred_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
        predict_time = time.time() - start

        # Collect performance metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_pred_prob) if y_pred_prob is not None else None

        # Print performance metrics
        print(model)
        print(f"\tTraining time: {train_time:.3f}s")
        print(f"\tPrediction time: {predict_time:.3f}s")
        print(f"\tAccuracy: {accuracy:.3f}")
        print(f"\tPrecision: {precision:.3f}")
        print(f"\tRecall: {recall:.3f}")
        print(f"\tF1 Score: {f1:.3f}")
        if roc_auc is not None:
            print(f"\tROC-AUC Score: {roc_auc:.3f}")
        print()

        # Append the ROC-AUC score to the results dataframe (if available)
        if roc_auc is not None:
            row = pd.DataFrame({'Model': [type(model).__name__], 'ROC-AUC Score': [roc_auc]})
            results_df = pd.concat([results_df, row], ignore_index=True)

        # Classification report and confusion matrix
        print(f"Classification Report for {type(model).__name__}:\n")
        print(classification_report(y_test, y_pred))

        # Plot confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
        plt.title(f'Confusion Matrix for {type(model).__name__}')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.show()

    except Exception as e:
        print(f"Model {model} failed: {e}")
        print()

# Compare performance metrics and select the best couple of models

In [ ]:
# Sort the DataFrame by ROC-AUC score
results_df = results_df.sort_values(by='ROC-AUC Score', ascending=False)

# Display the sorted ROC-AUC scores
print(results_df)

# Plot a bar chart for ROC-AUC scores
plt.figure(figsize=(10, 6))
plt.barh(results_df['Model'], results_df['ROC-AUC Score'], color='skyblue')
plt.xlabel('ROC-AUC Score')
plt.title('ROC-AUC Scores of Different Classifiers')
plt.gca().invert_yaxis()
plt.show()